In [10]:
import pandas as pd
import numpy as np
import torch
import re
import yaml
import dotenv
import sys
import os
sys.path.append(os.path.abspath('../..'))
from utils.metrics import *

In [11]:
dotenv.load_dotenv('../../.env')
SWEEP_EXP_DIR=os.getenv("SWEEP_EXP_DIR")
SWEEP_ANALYSIS_DIR=os.getenv("SWEEP_ANALYSIS_DIR")
LAPROTT5_OUTPUT=os.getenv("LAPROTT5_OUTPUT")

In [12]:
LEVEL_CLASSES = yaml.safe_load(open("../../datasets/final/hierarchical_label_set.yaml"))

In [13]:
testset = pd.read_csv('../../datasets/final/hou_testset.csv')

implicitly_multi = [
    "actin-filaments",
    "intermediate-filaments",
    "centrosome",
    "microtubules",
    "endosomes",
    "lysosomes",
    "peroxisomes"
    "lipid-droplets"
    ]
pattern = "|".join(map(re.escape, implicitly_multi))

single_testset = testset[~testset.level3.str.contains(";")]
single_testset.loc[single_testset.level2.str.contains(";"), "level2"] = pd.NA
single_testset.loc[
    (single_testset.level1.str.contains(";")) &
    ~((single_testset.level1.str.contains(pattern, na=False)) & (single_testset['level1'].str.count(";") == 1)), 
    "level1"] = pd.NA

multi_testset = testset[~testset.uniprot_id.isin(single_testset.uniprot_id)]
multi_testset.loc[~multi_testset.level2.str.contains(";"), "level2"] = pd.NA
multi_testset.loc[~multi_testset.level1.str.contains(";"), "level1"] = pd.NA


/tmp/ipykernel_48799/3458357824.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  single_testset.loc[single_testset.level2.str.contains(";"), "level2"] = pd.NA
/tmp/ipykernel_48799/3458357824.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  single_testset.loc[
/tmp/ipykernel_48799/3458357824.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  multi_testset.loc[~multi_testset.level2.str.contains(";"), "level2"] = pd.NA
/tmp/ipyk

In [14]:
avg_metrics = pd.read_csv(f"{SWEEP_ANALYSIS_DIR}/overall_metrics.csv")

In [15]:
#DROP DUPLICATE RUNS
idx = avg_metrics.groupby(
    ["exp_name",
    "category_level",
    "metadata_file",
    "clip_len",
    "agg_method",
    "mlp_dropout",
    "loss"
    ])["macro_ap"].idxmax()
avg_metrics = avg_metrics.loc[idx].reset_index(drop=True)

In [16]:
#Extract laprott5 results
laprott5_avg_metrics = avg_metrics[(avg_metrics.metadata_file=="hpa_uniprot_combined_trainset") &
                                    (avg_metrics.clip_len==1024) &
                                    (avg_metrics.exp_name=="ProtT5") & 
                                    (avg_metrics.agg_method=="LightAttentionPool") &
                                    (avg_metrics.loss=="BCEWithLogitsLoss") &
                                    (avg_metrics.mlp_dropout == 0.25)]
laprott5_avg_metrics = laprott5_avg_metrics.drop("acc", axis=1).rename({"acc_samples": "acc"}, axis=1)

In [17]:
laprott5_avg_metrics[laprott5_avg_metrics.category_level == f"level1"].run_id

2722    h7n0fcs3
Name: run_id, dtype: object

In [18]:
output_file = f"{LAPROTT5_OUTPUT}/LAProtT5_avg_metrics_single.csv"

if not os.path.exists(output_file):
    avg_metrics_single = []
    d = f"{SWEEP_EXP_DIR}/ProtT5_hpa_uniprot_combined_trainset"
    for level in [1,2,3]:
        run_id = laprott5_avg_metrics[laprott5_avg_metrics.category_level == f"level{level}"].run_id.to_list()[0]
        thresholds = np.load(f"{d}/{run_id}/all_thresholds.npy")
        preds_bin = []
        preds_all = []
        for i in range(5):
            path = f"{d}/{run_id}/fold_{i}/fold_{i}_test_predictions.csv"
            preds_df = pd.read_csv(path)
            preds_df = preds_df[preds_df.id.isin(single_testset.uniprot_id)]
            cols = preds_df.columns
            true_cols = [c for c in cols if "true" in c]
            pred_cols = [c for c in cols if "pred" in c]
            if i == 0:
                locations = np.array([c.split("_")[0] for c in true_cols])
                targets = np.array(preds_df[true_cols].to_numpy())
            else:
                assert (targets == preds_df[true_cols].to_numpy()).all()
                assert (locations == np.array([c.split("_")[0] for c in true_cols])).all()
                assert (locations == np.array([c.split("_")[0] for c in pred_cols])).all()
            preds = preds_df[pred_cols].to_numpy()
            preds = torch.sigmoid(torch.from_numpy(preds)).numpy()
            preds_all.append(preds)
            preds_bin.append((preds > thresholds[i]).astype(np.int16))
        preds_all = np.array(preds_all).mean(axis=0)
        preds_bin = (np.stack(preds_bin).mean(axis=0) > 0.5).astype(np.int16)


        metrics_dict, metrics_perclass, metrics_avg = get_all_metrics(targets, preds_all, preds_bin, LEVEL_CLASSES[f"level{level}"]
        )
        metrics_perclass["location"] = locations    
        metrics_perclass.to_csv(f"{LAPROTT5_OUTPUT}/LAProtT5_perclass_metrics_level{level}_single.csv")
        metrics_avg["level"] = level
        avg_metrics_single.append(metrics_avg)
    pd.concat(avg_metrics_single).to_csv(output_file)

In [19]:
output_file = f"{LAPROTT5_OUTPUT}/LAProtT5_avg_metrics_multi.csv"

if not os.path.exists(output_file):
    avg_metrics_multi = []
    d = "/scratch/groups/emmalu/seq2loc/sweep_experiments/ProtT5_hpa_uniprot_combined_trainset"
    for level in [1,2,3]:
        run_id = laprott5_avg_metrics[laprott5_avg_metrics.category_level == f"level{level}"].run_id.to_list()[0]
        thresholds = np.load(f"{d}/{run_id}/all_thresholds.npy")
        preds_bin = []
        preds_all = []
        for i in range(5):
            path = f"{d}/{run_id}/fold_{i}/fold_{i}_test_predictions.csv"
            preds_df = pd.read_csv(path)
            preds_df = preds_df[preds_df.id.isin(multi_testset.uniprot_id)]
            cols = preds_df.columns
            true_cols = [c for c in cols if "true" in c]
            pred_cols = [c for c in cols if "pred" in c]
            if i == 0:
                locations = np.array([c.split("_")[0] for c in true_cols])
                targets = np.array(preds_df[true_cols].to_numpy())
            else:
                assert (targets == preds_df[true_cols].to_numpy()).all()
                assert (locations == np.array([c.split("_")[0] for c in true_cols])).all()
                assert (locations == np.array([c.split("_")[0] for c in pred_cols])).all()
            preds = preds_df[pred_cols].to_numpy()
            preds = torch.sigmoid(torch.from_numpy(preds)).numpy()
            preds_all.append(preds)
            preds_bin.append((preds > thresholds[i]).astype(np.int16))
        preds_all = np.array(preds_all).mean(axis=0)
        preds_bin = (np.stack(preds_bin).mean(axis=0) > 0.5).astype(np.int16)


        metrics_dict, metrics_perclass, metrics_avg = get_all_metrics(targets, preds_all, preds_bin, LEVEL_CLASSES[f"level{level}"])
        metrics_perclass["location"] = locations    
        metrics_perclass.to_csv(f"{LAPROTT5_OUTPUT}/LAProtT5_perclass_metrics_level{level}_multi.csv")
        metrics_avg["category_level"] = f"level{level}"
        avg_metrics_multi.append(metrics_avg)
    pd.concat(avg_metrics_multi).to_csv(output_file)